<a href="https://colab.research.google.com/github/towardsai/ai-tutor-rag-system/blob/main/notebooks/14-Adding_Chat.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Install Packages and Setup Variables


In [ ]:
!pip install -q llama-index==0.14.0 openai==1.82.0 chromadb==1.0.12 llama-index-vector-stores-chroma==0.5.5 jedi==0.19.2 \
                llama-index-llms-openai==0.5.6 ipywidgets==8.1.5

In [ ]:
import os

# Set the following API Keys in the Python environment. Will be used later.
# os.environ["OPENAI_API_KEY"] = "<YOUR_OPENAI_KEY>"

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
except ImportError:
    os.environ.setdefault("OPENAI_API_KEY", os.environ.get("OPENAI_API_KEY", ""))

In [ ]:
# Allows running asyncio in environments with an existing event loop, like Jupyter notebooks.

import nest_asyncio

nest_asyncio.apply()

# Load Models


In [ ]:
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI
from llama_index.core import Settings

Settings.llm = OpenAI(model="gpt-4o-mini")
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")

# Load Indexes


In [ ]:
# Downloading Vector store from Hugging face hub
import shutil
from huggingface_hub import hf_hub_download

_local_vs = "/Users/jai/Documents/code-repo/ai-engineering-v2-notebooks-claude-skill/ai-engineering-v2-notebooks-claude-skill/notebooks/vectorstore.zip"
if os.path.exists(_local_vs):
    shutil.copy(_local_vs, "vectorstore.zip")
    vectorstore = "vectorstore.zip"
else:
    vectorstore = hf_hub_download(repo_id="jaiganesan/ai_tutor_knowledge", filename="vectorstore.zip", repo_type="dataset", local_dir=".")

In [ ]:
import zipfile
if not os.path.exists("ai_tutor_knowledge"):
    with zipfile.ZipFile("vectorstore.zip", 'r') as zf:
        zf.extractall(".")

In [ ]:
import chromadb
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import VectorStoreIndex

# Load the vector store from the local storage.
db = chromadb.PersistentClient(path="./ai_tutor_knowledge")
chroma_collection = db.get_or_create_collection("ai_tutor_knowledge")
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
vector_index = VectorStoreIndex.from_vector_store(vector_store)

# Display result


In [ ]:
# A simple function to show the response and the sources.
def display_res(response):
    print("Response:\n\t", response.response.replace("\n", ""))

    print("Sources:")
    if response.source_nodes:
        for src in response.source_nodes:
            print("\tNode ID\t", src.node_id)
            print("\tText\t", src.text)
            print("\tScore\t", src.score)
            print("\t" + "-_" * 20)
    else:
        print("\tNo sources used!")

# Chat Engine


In [ ]:
# define the chat_engine by using the index
chat_engine = vector_index.as_chat_engine()

In [ ]:
# First Question:
response = chat_engine.chat("Use the tool to answer, how does parameter efficient finetuning work?")

display_res(response)

In [ ]:
# Second Question:
response = chat_engine.chat("Could you tell me a joke?")
display_res(response)

In [ ]:
# Third Question: (check if it can recall previous interactions)
response = chat_engine.chat("What was the first question I asked?")
display_res(response)

In [ ]:
# Reset the session to clear the memory
chat_engine.reset()

In [ ]:
# Fourth Question: (don't recall the previous interactions.)
response = chat_engine.chat("What was the first question I asked?")
display_res(response)

# Streaming


In [ ]:
# Stream the words as soon as they are available instead of waiting for the model to finish generation.
streaming_response = chat_engine.stream_chat(
    "Write a paragraph explaining how RAG and PEFT work, and highlight the differences between them."
)
streaming_response.print_response_stream()

## Condense Question


Enhance the input prompt by looking at the previous chat history along with the current question. The refined prompt can then be used to fetch the nodes.


In [ ]:
# Define GPT-4o model that will be used by the chat engine to improve the query.
gpt5 = OpenAI(model="gpt-4o")

In [ ]:
chat_engine = vector_index.as_chat_engine(
    chat_mode="condense_question", llm=gpt5, verbose=True
)

In [ ]:
response = chat_engine.chat(
    "How does Retrieval-Augmented Generation (RAG) work, and which problem does it solve?"
)
display_res(response)

## ReAct


In [ ]:
from llama_index.core.agent.workflow import ReActAgent
from llama_index.core.workflow import Context
from llama_index.core.tools import QueryEngineTool

query_engine = vector_index.as_query_engine()

tool = QueryEngineTool.from_defaults(
    query_engine=query_engine,
    name="ReAct Agent",
    description="Answer questions using the vector index; pass plain text queries.",
)

agent = ReActAgent(
    tools=[tool],
    verbose=True

)

# context to hold this session/state
ctx = Context(agent)

import asyncio as _asyncio

async def _run_react():
    return await agent.run("Which company developed Claude 3.5 Sonnet, and what is its primary application?", ctx=ctx, max_iterations=4)

In [ ]:
response = _asyncio.run(_run_react())
print(str(response))

## Optional: Interactive Chat Interface

Use the widget below to have a multi-turn conversation with the chat engine. Type your message and press "Send" to get a response.

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display

    # Fresh chat engine for the demo
    _demo_chat_engine = vector_index.as_chat_engine()

    msg_input = widgets.Text(
        placeholder="Type your question here...",
        description="You:",
        layout=widgets.Layout(width="70%"),
        style={"description_width": "initial"},
    )
    send_btn = widgets.Button(description="Send", button_style="success")
    reset_btn = widgets.Button(description="Reset Chat", button_style="warning")
    history_out = widgets.Output(layout=widgets.Layout(border="1px solid #ccc", min_height="100px", padding="8px"))

    def on_send(b):
        msg = msg_input.value.strip()
        if not msg:
            return
        msg_input.value = ""
        with history_out:
            print(f"You: {msg}")
            r = _demo_chat_engine.chat(msg)
            print(f"Bot: {r.response}\n")

    def on_reset(b):
        _demo_chat_engine.reset()
        history_out.clear_output()
        with history_out:
            print("[Chat history cleared]")

    send_btn.on_click(on_send)
    reset_btn.on_click(on_reset)
    display(widgets.VBox([widgets.HBox([msg_input, send_btn, reset_btn]), history_out]))
except ImportError:
    print("ipywidgets not installed. Run: pip install ipywidgets")